# L13 · Meta-learners y efectos heterogéneos

**S13 · Lectura: Cap. 12 · 40–55 min**

## La pregunta

¿Dos personas con el mismo riesgo tienen el mismo efecto?

## Qué harás

Comparar formas de estimar CATE.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** CATE, PEHE y placebo sobre IHDP.

> **Idea clave:** El riesgo predicho dice quién puede tener un mal resultado; el CATE dice a quién le cambia actuar.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import Lasso, LogisticRegression
rng = setup()
N_REP = R(3, 20)
print("repeticiones por tamaño:", N_REP)

## 1. El mundo y el objetivo (§12.1)
Mundo del capítulo (Listado 12.1): $X\sim U(-1,1)^{10}$, propensity $e=\sigma(1+0{,}5X_0)$, $\mu_0=X_2+0{,}5X_3$ y efecto **heterogéneo** $\tau(x)=2+3X_0-X_1^2$.

Los cuatro meta-learners, en una línea cada uno:
- **S:** un solo modelo $\hat\mu(X,D)$; $\hat\tau=\hat\mu(x,1)-\hat\mu(x,0)$.
- **T:** dos modelos, uno por brazo: $\hat\tau=\hat\mu_1(x)-\hat\mu_0(x)$.
- **X:** imputa efectos individuales con el modelo del otro brazo, los regresa sobre $X$ y mezcla con $\hat e$.
- **R:** regresa los residuos $\tilde Y$ sobre $\tilde D$ con pesos $\tilde D^2$ (Robinson, Cap. 10 con $\tau$ variable).

In [ ]:
def make_dgp(n=2000, p=10, seed=0, a=1.0, b=0.5):
    r = np.random.default_rng(seed); X = r.uniform(-1, 1, (n, p)); e = 1 / (1 + np.exp(-a - b * X[:, 0]))
    D = (r.uniform(size=n) < e).astype(int); tau = 2 + 3 * X[:, 0] - X[:, 1] ** 2; mu0 = X[:, 2] + .5 * X[:, 3]
    Y = mu0 + D * tau + r.standard_normal(n) * .5
    return Y, D, X, tau
Y, D, X, tau_true = make_dgp(2000, seed=42)
Yt, Dt, Xt, tau_t = make_dgp(2000, seed=7)          # conjunto de PRUEBA con otra semilla: evaluamos fuera de muestra
ML = lambda: HistGradientBoostingRegressor(max_iter=100, learning_rate=.1, max_depth=4, random_state=0)
CL = lambda: HistGradientBoostingClassifier(max_iter=60, learning_rate=.1, max_depth=3, random_state=0)
print("tasa de tratamiento:", D.mean().round(3), "· τ medio verdadero:", tau_true.mean().round(3))

## 2. S y X ya hechos; T y R los completas tú
Aquí tienes `s_learner` y `x_learner`. 

### Completa tú
- `t_learner(Y, D, X, Xnew, make)`: dos modelos, uno por brazo.
- `r_learner(Y, D, X, Xnew, make_reg, make_clf, K=5)`: (1) con **cross-fitting** obtén $\hat m(X)=E[Y|X]$ y $\hat e(X)$; (2) residuos $\tilde Y=Y-\hat m$, $\tilde D=D-\hat e$; (3) ajusta un regresor de $\tilde Y/\tilde D$ sobre $X$ con `sample_weight` $=\tilde D^2$ y predice en `Xnew`.

In [ ]:
def s_learner(Y, D, X, Xnew, make):
    m = make().fit(np.column_stack([X, D]), Y)
    return m.predict(np.column_stack([Xnew, np.ones(len(Xnew))])) - m.predict(np.column_stack([Xnew, np.zeros(len(Xnew))]))

def x_learner(Y, D, X, Xnew, make):
    m1, m0 = make().fit(X[D == 1], Y[D == 1]), make().fit(X[D == 0], Y[D == 0])
    g1 = make().fit(X[D == 1], Y[D == 1] - m0.predict(X[D == 1]))
    g0 = make().fit(X[D == 0], m1.predict(X[D == 0]) - Y[D == 0])
    e = LogisticRegression(max_iter=500).fit(X, D).predict_proba(Xnew)[:, 1]
    return e * g0.predict(Xnew) + (1 - e) * g1.predict(Xnew)

In [ ]:
# ✏️ Completa tú — t_learner y r_learner
# Pista: T: dos .fit, uno con X[D==1] y otro con X[D==0]. R: el regresor final usa y = ry/rd y sample_weight = rd**2.
def t_learner(Y, D, X, Xnew, make):
    """Un modelo por brazo; devuelve el CATE estimado en Xnew."""
    raise NotImplementedError("completa t_learner")

def r_learner(Y, D, X, Xnew, make_reg, make_clf, K=5):
    """R-learner con cross-fitting (kfold(n, K, SEED)). Devuelve el CATE estimado en Xnew."""
    raise NotImplementedError("completa r_learner")

In [ ]:
def rmse(h, t): return float(np.sqrt(np.mean((h - t) ** 2)))
Xs, Ys, Ds = X[:600], Y[:600], D[:600]; Xn = Xt[:300]; tn = tau_t[:300]
h_t = t_learner(Ys, Ds, Xs, Xn, ML); h_r = r_learner(Ys, Ds, Xs, Xn, ML, CL)
lab_check("t_learner y r_learner (correlación con el τ verdadero y escala)", lambda: (np.corrcoef(h_t, tn)[0, 1] > .8, np.corrcoef(h_r, tn)[0, 1] > .7, abs(h_t.mean() - tn.mean()) < .6, abs(h_r.mean() - tn.mean()) < .8),
          [((), (True, True, True, True), 0)])

## 3. La tabla del capítulo: RMSE y sesgo (§12.3)
Se evalúa en el conjunto de **prueba** (el libro lo hace en la misma muestra; fuera de muestra es lo honesto). El τ verdadero solo existe porque simulamos.

In [ ]:
def todos(Y, D, X, Xn, make=None, mc=None):
    make, mc = make or ML, mc or CL
    return {"S": s_learner(Y, D, X, Xn, make), "T": t_learner(Y, D, X, Xn, make), "X": x_learner(Y, D, X, Xn, make), "R": r_learner(Y, D, X, Xn, make, mc)}
H = todos(Y, D, X, Xt)
# EVALUATION-CELL
tab = pd.DataFrame({"RMSE": {k: rmse(v, tau_t) for k, v in H.items()}, "sesgo": {k: float(np.mean(v - tau_t)) for k, v in H.items()}})
show(tab.round(3))

**Mira el resultado.** Si un meta-learner tiene RMSE bajo pero sesgo grande (en media), ¿qué ocurre al usarlo para estimar el ATE?

**Respuesta:**

Que el ATE queda corrido aunque los CATE individuales parezcan ordenados. RMSE y sesgo miden cosas distintas: el RMSE mezcla varianza y sesgo; el sesgo medio es lo que afecta al efecto promedio. Para decisiones basadas en **ranking** importa el orden (Qini, L11); para estimar un efecto total importa el sesgo.



## 4. Señal de alerta: el S-learner que colapsa (Listado 12.2)
Si el S-learner usa un modelo **regularizado**, el término de $D$ se encoge como cualquier otro coeficiente. Con un Lasso fuerte el efecto del tratamiento puede quedar exactamente en 0: todos los CATE valen 0 y ningún error de código lo avisa.

In [ ]:
def s_lasso(alpha):
    Z = np.column_stack([X, D]); m = Lasso(alpha=alpha, max_iter=5000).fit(Z, Y)
    h = m.predict(np.column_stack([Xt, np.ones(len(Xt))])) - m.predict(np.column_stack([Xt, np.zeros(len(Xt))]))
    return h, m.coef_[-1]
filas = []
for al in (0.001, 0.05, 0.2, 0.5):
    h, c = s_lasso(al); filas.append((al, c, float((np.abs(h) < 0.01).mean())))
show(pd.DataFrame(filas, columns=["alpha (Lasso)", "coef. de D", "fracción de CATE ≈ 0"]).set_index("alpha (Lasso)").round(3))
h5, _ = s_lasso(0.5)
h_tl = t_learner(Y, D, X, Xt, lambda: Lasso(alpha=0.5, max_iter=5000))             # el MISMO Lasso, pero en un T-learner
print("fracción de CATE ≈ 0 con el mismo Lasso (alpha=0,5) en un T-learner:", round(float((np.abs(h_tl) < 0.01).mean()), 3))
# EVALUATION-CELL
print("RMSE del S-learner con alpha=0,5:", round(rmse(h5, tau_t), 3), "· del T-learner con Lasso:", round(rmse(h_tl, tau_t), 3), "· (τ verdadero tiene media", round(tau_t.mean(), 2), ")")

**Mira el resultado.** ¿Por qué el T, el X o el R-learner no sufren este colapso con el mismo Lasso?

**Respuesta:**

Porque el efecto **no es un coeficiente penalizable**: en T se obtiene restando predicciones de dos modelos que sí ajustan Y por brazo; en X y R el modelo final se entrena directamente sobre efectos imputados o residualizados, no sobre Y con D como una variable más entre muchas. Moraleja: si usas un S-learner, el modelo debe poder representar el efecto sin penalizarlo (o debes incluir interacciones y no regularizar D).



## 5. ¿Cambia el ranking con $n$ y con el learner base? (§12.5, ejercicio)
Repetimos la comparación con $n\in\{500,1000,2000,5000\}$ y varias réplicas, primero con boosting y luego con un learner base **lineal** (Ridge/logística).

In [ ]:
filas = []
for n_ in (500, 1000, 2000, 5000):
    acum = {k: [] for k in "STXR"}
    for s in range(N_REP):
        Y_, D_, X_, _ = make_dgp(n_, seed=100 + s); Xn_, tn_ = make_dgp(2000, seed=900 + s)[2:]
        for k, v in todos(Y_, D_, X_, Xn_).items(): acum[k].append(rmse(v, tn_))
    filas.append({"n": n_, **{k: np.mean(v) for k, v in acum.items()}})
rank_n = pd.DataFrame(filas).set_index("n"); rank_n["mejor"] = rank_n[list("STXR")].idxmin(axis=1)
show(rank_n.round(3))

In [ ]:
from sklearn.linear_model import RidgeCV
RIDGE = lambda: RidgeCV(alphas=np.logspace(-3, 2, 6)); LOGIT = lambda: LogisticRegression(max_iter=500)
filas = []
for n_ in (500, 5000):
    acum = {k: [] for k in "STXR"}
    for s in range(N_REP):
        Y_, D_, X_, _ = make_dgp(n_, seed=100 + s); Xn_, tn_ = make_dgp(2000, seed=900 + s)[2:]
        for k, v in todos(Y_, D_, X_, Xn_, RIDGE, LOGIT).items(): acum[k].append(rmse(v, tn_))
    filas.append({"n": n_, **{k: np.mean(v) for k, v in acum.items()}})
show(pd.DataFrame(filas).set_index("n").round(3))

**Mira el resultado.** Compara las dos tablas. ¿Qué se mantiene y qué cambia al pasar de boosting a un learner lineal?

**Respuesta:**

Con boosting, el X-learner queda primero en todos los tamaños de esta simulación y el R-learner último; el orden entre S y T cambia con $n$. Con un learner **lineal** el S-learner se vuelve inservible (un único coeficiente para $D$ es un efecto constante: no puede representar $3X_0-X_1^2$) y T, X y R se igualan, limitados por no poder ajustar el término cuadrático. Conclusión: "el mejor meta-learner" depende del **learner base**, de $n$ y del mundo; no se elige por reputación, se **valida** (Qini y placebos de L11, y simulación cuando se puede).



## 6. Propensity extremo (§12.5, ejercicio)
Con $e=\sigma(4X_0)$ casi no hay controles donde $X_0$ es alto (y casi no hay tratados donde es bajo). Comparamos las dos situaciones.

In [ ]:
def compara(a, b, n=2000, reps=N_REP):
    acum = {k: [] for k in "STXR"}; fuera = []
    for s in range(reps):
        Y_, D_, X_, _ = make_dgp(n, seed=300 + s, a=a, b=b); Xn_, tn_ = make_dgp(2000, seed=950 + s, a=a, b=b)[2:]
        e_ = LogisticRegression(max_iter=500).fit(X_, D_).predict_proba(X_)[:, 1]; fuera.append(((e_ < .05) | (e_ > .95)).mean())
        for k, v in todos(Y_, D_, X_, Xn_).items(): acum[k].append(rmse(v, tn_))
    return {**{k: np.mean(v) for k, v in acum.items()}, "% ê fuera [.05,.95]": 100 * np.mean(fuera)}
show(pd.DataFrame({"moderado (b=0,5)": compara(1.0, .5), "extremo (b=4)": compara(0.0, 4.0)}).round(3))

**Mira el resultado.** ¿Qué cambia entre las dos columnas y qué deberías hacer ante el segundo caso?

**Respuesta:**

Con propensity extremo todos empeoran, y los métodos que **dependen del propensity** (X, R) se vuelven más inestables mientras que S y T extrapolan sin aviso (en la simulación, el R-learner, que divide por residuos de tratamiento cercanos a 0, es el que más se deteriora). La respuesta no es elegir otro learner sino **mirar el solapamiento** (L10): recortar, reportar la región donde el CATE es identificable y no extrapolar fuera de ella.



## 7. IHDP con verdad: PEHE en datos de prueba
En `real/ihdp` sí existe el efecto individual (es semisintético). Entrenamos en `train` y evaluamos en `test`.

In [ ]:
sc = load_scenario("real/ihdp"); sc.card()
X1, Y1, D1 = sc.xyt("train"); X2, _, _ = sc.xyt("test")
Hi = todos(Y1, D1.astype(int), X1, X2)
# EVALUATION-CELL
tt = sc.tau("test")
if tt is not None:
    pehe = pd.DataFrame({"PEHE (raíz del ECM)": {k: rmse(v, tt) for k, v in Hi.items()}, "error del ATE": {k: float(v.mean() - tt.mean()) for k, v in Hi.items()}})
    show(pehe.round(3)); print("filas de prueba:", len(tt), "(pocas ⇒ las diferencias entre learners pueden ser ruido)")
else:
    print("este escenario no trae el τ verdadero en tu copia del benchmark")

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 8. Supuesto y bitácora

In [ ]:
mejor_2000 = rank_n.loc[2000, "mejor"]
fila_b = bitacora(
    pregunta="¿Qué meta-learner estima mejor el CATE en el mundo del capítulo y cuánto depende esa respuesta de n y del solapamiento?",
    metodo="S, T, X y R (R con cross-fitting K=5) con boosting; RMSE y sesgo fuera de muestra",
    estimando="CATE τ(x) = 2 + 3·x0 − x1²",
    numero_ic=f"mejor con n=2000: {mejor_2000} (RMSE {rank_n.loc[2000, mejor_2000]:.3f}); ganador por n: " + ", ".join(f"{i}→{m}" for i, m in rank_n['mejor'].items()),
    supuesto=supuesto("no observable", "declarado y acotado", "no confusión condicional dentro de cada subgrupo de X; no se puede contrastar con los datos"),
    decision="continuar", lab="L13")

---
## Qué queda para el libro
Quedan en el libro: el caso NSW (`nsw_mixtape` no está en el benchmark), el DR-learner y el CausalForestDML del Listado 12.1. Aquí el R-learner se hace a mano con cross-fitting.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*